# About the Dataset 
This dataset was taken from Kaggle provided by the Specialized Center for Endocrinology and Diabetes at Al-Kindy Teaching Hospital in Iraq. The dataset has 14 features and 800 observations. Features are medical metrics relevant to diabetes diagnosis: age, bmi, cholesterol, etc. The target variable is a binary classification of diabetic and not-diabetic.  

## Data Collection
Imported Data from Kaggle, extracting the Dataset as a CSV.

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("aravindpcoder/diabetes-dataset")

# print("Path to dataset files:", path)

In [ ]:
# import shutil
# import os

# source_path = r"C:\Users\siver\.cache\kagglehub\datasets\aravindpcoder\diabetes-dataset\versions\1\Dataset of Diabetes .csv"

# destination_dir = os.getcwd()

# shutil.move(source_path, destination_dir)

# print(f"File moved to {destination_dir}")

## Evaluate Data

In [ ]:
import pandas as pd
df = pd.read_csv("Dataset of Diabetes .csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.describe() # Check distribution and range of values

In [ ]:
df['CLASS'].value_counts()

## Feature Engineering

In [ ]:
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
df = df[df['CLASS'] != 'P']
df

In [ ]:
class_mapper = {
    'Y': 1,
    'N': 0
}

df['CLASS'] = df['CLASS'].str.strip() # Remove whitespace

df['CLASS'] = df['CLASS'].map(class_mapper) # Change to numerical values

df['CLASS'].value_counts()

In [ ]:
df = df.drop(columns=['ID', 'No_Pation']) # Drop unneccessary columns

class_col = df.pop('CLASS') # Put target CLASS in first column
df.insert(0, 'CLASS', class_col)

df.head()

In [ ]:
gender_mapper = {
    'M': 0,
    'F': 1
}

df['Gender'] = df['Gender'].str.strip().str.upper() # Remove whitespace, convert to uppercase

df['Gender'] = df['Gender'].map(gender_mapper) # Change to numerical values

df['Gender'].value_counts()

In [ ]:
df.head()

In [ ]:
df.info() # Check for missing values

In [ ]:
import matplotlib.pyplot as plt

df.hist(figsize=(12, 8)) # Check distributions
plt.show()

In [ ]:
df.boxplot(figsize=(12, 6)) # Check for outliers

In [ ]:
df['Cr'].sort_values(ascending=False).head()

In [ ]:
df[df['Cr'] == 800]

In [ ]:
df.columns[(df == 0).any()] # Checking columns that are measurements for zeros

In [ ]:
df.columns[(df == 0.0).any()]

In [ ]:
df[df['Chol'] == 0.0] # removing zeros in Chol, can't have zero cholesterol

In [ ]:
df = df[df['Chol'] != 0.0]
df[df['Chol'] == 0.0]

## Select and Train Model
We chose sklearn's LogisticRegression as our model. Training size is 80%, test and validation sets were 10% each. We maintained a stratified sample to help with class imbalance.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
train, test_and_validate = train_test_split(df, test_size=0.2, random_state=42, stratify=df['CLASS'])

In [ ]:
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['CLASS'])

In [ ]:
print(train.shape)
print(test.shape)
print(validate.shape)

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

# Separate features and target
X_train = train.drop(columns='CLASS')
y_train = train['CLASS']

X_validate = validate.drop(columns='CLASS')
y_validate = validate['CLASS']

X_test = test.drop(columns='CLASS')
y_test = test['CLASS']

# Scale features using only training data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validate_scaled = scaler.transform(X_validate)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(random_state=42, max_iter=1000) # Maximum iterations set higher because scaling is used
model.fit(X_train_scaled, y_train)

## Evaluate Model

In [ ]:
from sklearn.metrics import confusion_matrix

y_val_pred = model.predict(X_validate_scaled)

matrix = confusion_matrix(y_validate, y_val_pred)

df_confusion = pd.DataFrame(
    matrix,
    index=['Predicted Normal', 'Predicted Diabetes'],
    columns=['Actual Normal', 'Actual Diabetes']
)

df_confusion

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(df_confusion, 
            annot=True, 
            fmt='d',
            cmap='Blues',
            cbar=True,
            linewidths=.5,
            linecolor='gray') 

plt.title("Confusion Matrix - Validation Set Performance")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

coefficients = model.coef_[0]

feature_names = X_train.columns

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Absolute_Impact': np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(by='Absolute_Impact', ascending=False).reset_index(drop=True)

print("Top 5 Most Significant Features:")
display(feature_importance[['Feature', 'Coefficient']].head())

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Coefficient', 
    y='Feature', 
    data=feature_importance, 
    palette='coolwarm',
    hue='Feature',
    legend=False
)

plt.title("Logistic Regression Feature Significance", fontsize=14)
plt.xlabel("Coefficient Value (Impact on Diabetes Classification)", fontsize=12)
plt.ylabel("Medical Metric", fontsize=12)

plt.axvline(x=0, color='black', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc

TN, FP, FN, TP = confusion_matrix(y_validate, y_val_pred).ravel()

print(f"True Negative (TN) : {TN}")
print(f"False Positive (FP): {FP}")
print(f"False Negative (FN): {FN}")
print(f"True Positive (TP) : {TP}")

In [ ]:
# Sensitivity, hit rate, recall, or true positive rate of validation set
Sensitivity  = float(TP)/(TP+FN)*100
print(f"Sensitivity or TPR: {Sensitivity}%")  
print(f"There is a {Sensitivity}% chance of detecting patients with diabetes to actually have diabetes")

In [ ]:
# Specificity or true negative rate of validation set
Specificity  = float(TN)/(TN+FP)*100
print(f"Specificity or TNR: {Specificity}%") 
print(f"There is a {Specificity}% chance of detecting non-diabetic patients as non-diabetic.")

In [ ]:
# Precision or positive predictive value of validation set
Precision = float(TP)/(TP+FP)*100
print(f"Precision: {Precision}%")  
print(f"You have diabetes, and the probablity of that is  {Precision}%")

In [ ]:
# Negative predictive value of validation set
NPV = float(TN)/(TN+FN)*100
print(f"Negative Predictive Value: {NPV}%") 
print(f"You don't have a diabetes, but there is a {NPV}% chance that is incorrect" )

In [ ]:
# Fall out or false positive rate of validation set
FPR = float(FP)/(FP+TN)*100
print( f"False Positive Rate: {FPR}%") 
print( f"There is a {FPR}% chance that this positive result is incorrect.")

In [ ]:
# False negative rate of validation set
FNR = float(FN)/(TP+FN)*100
print(f"False Negative Rate: {FNR}%") 
print(f"There is a {FNR}% chance that this negative result is incorrect.")

In [ ]:
# False discovery rate of validation set
FDR = float(FP)/(TP+FP)*100
print(f"False Discovery Rate: {FDR}%" )
print(f"You have diabetes, but there is a {FDR}% chance this is incorrect.")

In [ ]:
# Overall accuracy of validation set
ACC = float(TP+TN)/(TP+FP+FN+TN)*100
print(f"Accuracy: {ACC}%")

In [ ]:
print(f"Sensitivity or TPR: {Sensitivity}%")    
print(f"Specificity or TNR: {Specificity}%") 
print(f"Precision: {Precision}%")   
print(f"Negative Predictive Value: {NPV}%")  
print( f"False Positive Rate: {FPR}%") 
print(f"False Negative Rate: {FNR}%")  
print(f"False Discovery Rate: {FDR}%" )
print(f"Accuracy: {ACC}%") 

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_validate, y_val_pred)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")

ax2 = plt.gca().twinx()
# ax2.plot(fpr, thresholds, markeredgecolor='r', linestyle='dashed', color='r')
# ax2.set_ylabel('Threshold', color='r')
ax2.set_xlim([fpr[0], fpr[-1]])

plt.show()

## Tune Model

In [ ]:
# Hyperparameter grid of different values
param_grid = [
    {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': None},
    {'C': 0.1,  'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': None},
    {'C': 1.0,  'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': None},
    {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': 'balanced'},
    {'C': 0.1,  'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': 'balanced'},
    {'C': 1.0,  'penalty': 'l2', 'solver': 'lbfgs', 'class_weight': 'balanced'},
]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
results = []

# Tune models with different parameters
for params in param_grid:
    model = LogisticRegression(
        C=params['C'],
        penalty=params['penalty'],
        solver=params['solver'],
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_val_pred = model.predict(X_validate_scaled)
    # Probability that class is 1
    y_val_prob = model.predict_proba(X_validate_scaled)[:, 1]

    # Results of tuning
    results.append({
        'C': params['C'],
        'penalty': params['penalty'],
        'solver': params['solver'],
        'val_accuracy': accuracy_score(y_validate, y_val_pred),
        'val_f1': f1_score(y_validate, y_val_pred),
        'val_auc': roc_auc_score(y_validate, y_val_prob)
    })

# Results DataFrame sorted by AUC, best values first
results_df = pd.DataFrame(results).sort_values(by='val_accuracy', ascending=False)
results_df

In [ ]:
# Train the model with the best hyperparameters
best_model = LogisticRegression(
    C=results_df.iloc[0]['C'],
    penalty=results_df.iloc[0]['penalty'],
    solver=results_df.iloc[0]['solver'],
    max_iter=1000,
    random_state=42
)

best_model.fit(X_train_scaled, y_train)

In [ ]:
y_test_pred = best_model.predict(X_test_scaled)

matrix = confusion_matrix(y_test, y_test_pred)

df_confusion = pd.DataFrame(
    matrix, 
    index=['Predicted Normal', 'Predicted Diabetes'],
    columns=['Actual Normal', 'Actual Diabetes']
)

df_confusion

Graphed the Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(df_confusion, 
            annot=True, 
            fmt='d',
            cmap='Blues',
            cbar=True,
            linewidths=.5,
            linecolor='gray') 

plt.title("Confusion Matrix - Tuned Model Performance")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc

TN, FP, FN, TP = confusion_matrix(y_test, y_test_pred).ravel()

print(f"True Negative (TN) : {TN}")
print(f"False Positive (FP): {FP}")
print(f"False Negative (FN): {FN}")
print(f"True Positive (TP) : {TP}")

In [ ]:
# Sensitivity, hit rate, recall, or true positive rate
Sensitivity  = float(TP)/(TP+FN)*100
print(f"Sensitivity or TPR: {Sensitivity}%")  
print(f"There is a {Sensitivity}% chance of detecting patients with diabetes to actually have diabetes")

In [ ]:
# Specificity or true negative rate
Specificity  = float(TN)/(TN+FP)*100
print(f"Specificity or TNR: {Specificity}%") 
print(f"There is a {Specificity}% chance of detecting non-diabetic patients as non-diabetic.")

In [ ]:
# Precision or positive predictive value
Precision = float(TP)/(TP+FP)*100
print(f"Precision: {Precision}%")  
print(f"You have diabetes, and the probablity of that is  {Precision}%")

In [ ]:
# Negative predictive value
NPV = float(TN)/(TN+FN)*100
print(f"Negative Predictive Value: {NPV}%") 
print(f"You don't have a diabetes, but there is a {NPV}% chance that is incorrect" )

In [ ]:
# Fall out or false positive rate
FPR = float(FP)/(FP+TN)*100
print( f"False Positive Rate: {FPR}%") 
print( f"There is a {FPR}% chance that this positive result is incorrect.")

In [ ]:
# False negative rate
FNR = float(FN)/(TP+FN)*100
print(f"False Negative Rate: {FNR}%") 
print(f"There is a {FNR}% chance that this negative result is incorrect.")

In [ ]:
# False discovery rate
FDR = float(FP)/(TP+FP)*100
print(f"False Discovery Rate: {FDR}%" )
print(f"You have diabetes, but there is a {FDR}% chance this is incorrect.")

In [ ]:
# Overall accuracy
ACC = float(TP+TN)/(TP+FP+FN+TN)*100
print(f"Accuracy: {ACC}%") 

In [ ]:
print(f"Sensitivity or TPR: {Sensitivity}%")    
print(f"Specificity or TNR: {Specificity}%") 
print(f"Precision: {Precision}%")   
print(f"Negative Predictive Value: {NPV}%")  
print( f"False Positive Rate: {FPR}%") 
print(f"False Negative Rate: {FNR}%")  
print(f"False Discovery Rate: {FDR}%" )
print(f"Accuracy: {ACC}%") 

AUC-ROC

In [ ]:
y_test = test['CLASS']
y_test_prob = best_model.predict_proba(X_test_scaled)[:, 1]
print("Test AUC", roc_auc_score(y_test, y_test_prob))

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")

ax2 = plt.gca().twinx()
# ax2.plot(fpr, thresholds, markeredgecolor='r', linestyle='dashed', color='r')
# ax2.set_ylabel('Threshold', color='r')
ax2.set_xlim([fpr[0], fpr[-1]])

plt.show()